# Factual-consistency evaluation: AlignScore and SummaC-Conv

**Run in Google Colab:** select a GPU runtime if available, then run the new cells in order. The original notebook is retained at the end as a reference; **do not run its cells** as part of the comparison.

This notebook compares pretrained AlignScore-base and SummaC-Conv on the summarization portions of RAGTruth and HaluEval, and on FaithBench. It scores the same examples for both methods. `1` means an annotated or benchmark-designated hallucination; `0` means no such annotation. Start with the small pilot settings and increase them after checking the outputs.

This is an **evaluation on new datasets**, not an exact reproduction of the papers' published benchmark results. For reproduction, additionally run the exact original benchmark, model, split and metric. Dataset labels are not fully equivalent. See the label notes below.

## Mount Google Drive and create a results folder
Run this cell first and follow Colab’s authorization prompt. Each run gets a timestamped folder in `My Drive/EECS_6412_factuality_results` so earlier results are not overwritten. Change `DRIVE_BASE` below if you prefer another folder. The notebook itself should be saved using **File → Save a copy in Drive**; mounting Drive saves experiment files, not the open notebook automatically.

Partial scoring files are written every 10 examples. To keep an interrupted run, do not rerun the mount/setup cell until you have noted its folder; a new run creates a new folder. Scoring does not automatically resume from a partial file.

In [1]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import os

drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/EECS_6412_factuality_results')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
RUN_DIR = DRIVE_BASE / datetime.now().strftime('run_%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.environ['FACTUALITY_RESULTS_DIR'] = str(RUN_DIR)
print('Results will be saved to:', RUN_DIR)

Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/EECS_6412_factuality_results/run_20261028_171130_261670


## 1. Setup
Run in a fresh Colab runtime. The isolated Python 3.10 environment follows the versions in your original notebook and avoids modifying Colab's installed packages. The first run downloads models and datasets; it may take several minutes. A GPU is recommended; CPU runs can be slow. If a download or installation fails, inspect the error before proceeding.

In [2]:
import subprocess, sys
from pathlib import Path

def run(command):
    subprocess.run(command, check=True)

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
ENV = Path('/content/alignscore-env')
PYTHON = str(ENV / 'bin/python')
if not ENV.exists():
    run(['uv', 'venv', '--python', '3.10', str(ENV)])
run(['uv', 'pip', 'install', '--python', PYTHON,
    'torch==1.13.1', 'numpy==1.24.4', 'transformers==4.30.2',
    'pytorch-lightning==1.9.5', 'torchmetrics==0.11.4',
    'spacy==3.7.5', 'nltk==3.8.1', 'scikit-learn==1.3.2',
    'pandas==2.0.3', 'huggingface-hub==0.25.2', 'protobuf==3.20.0',
    'tensorboard==2.14.0', 'datasets==2.14.7', 'pyarrow==14.0.2',
    'jsonlines==2.0.0', 'tqdm', 'setuptools==65.5.0', 'sentencepiece'])
run(['uv', 'pip', 'install', '--python', PYTHON, '--no-deps',
     'git+https://github.com/yuh-zha/AlignScore.git', 'summac==0.0.4'])
run(['uv', 'pip', 'install', '--python', PYTHON,
     'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.0/en_core_web_sm-3.7.0-py3-none-any.whl'])
run([PYTHON, '-c', "import nltk; nltk.download('punkt', quiet=True)"])
print('Environment ready:', PYTHON)

Environment ready: /content/alignscore-env/bin/python


In [3]:
import subprocess
from pathlib import Path

# Install BERTScore in the SAME environment as AlignScore and SummaC.
subprocess.run(
    [
        "uv", "pip", "install",
        "--python", PYTHON,
        "bert-score==0.3.13",
    ],
    check=True,
)

# Download the authors' BARTScore implementation.
BARTSCORE_DIR = Path("/content/BARTScore")

if not BARTSCORE_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/neulab/BARTScore.git",
            str(BARTSCORE_DIR),
        ],
        check=True,
    )

print("BERTScore installed.")
print("BARTScore repository:", BARTSCORE_DIR)

BERTScore installed.
BARTScore repository: /content/BARTScore


In [4]:
import subprocess
import os

# Let's run an imports check inside the isolated environment with a headless backend set.
check_code = """
try:
    import sentencepiece
    import transformers
    import torch
    import pytorch_lightning
    import torchmetrics
    import spacy
    import nltk
    import sklearn
    import pandas
    import datasets
    import pyarrow
    import jsonlines
    import tqdm
    import alignscore
    import summac
    import bert_score
    import sys
    sys.path.insert(0, '/content/BARTScore')
    import bart_score
    print("SUCCESS: All imports loaded successfully inside the virtual environment!")
except ImportError as e:
    print(f"FAILURE: Missing dependency: {e}")
    sys.exit(1)
"""

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

try:
    res = subprocess.run([PYTHON, "-c", check_code], capture_output=True, text=True, env=env)
    print(res.stdout)
    if res.returncode != 0:
        print(res.stderr)
except Exception as e:
    print("Error running check:", e)

SUCCESS: All imports loaded successfully inside the virtual environment!



## 2. Download the datasets
Each repository's exact commit is recorded. Rerunning in an existing Colab session reuses the clone. For the final report record the commits and environment file from the results folder.

In [5]:
REPOS = {
 'RAGTruth': 'https://github.com/ParticleMedia/RAGTruth.git',
 'HaluEval': 'https://github.com/RUCAIBox/HaluEval.git',
 'FaithBench': 'https://github.com/vectara/FaithBench.git',
}
for name, url in REPOS.items():
    path = Path('/content') / name
    if not path.exists():
        run(['git', 'clone', '--depth', '1', url, str(path)])
    commit = subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip()
    print(name, commit)

RAGTruth c103204b9ce28d6bbad859304bf30de72b8ed8fe
HaluEval b7253db3cdaa0ab2c382f92b26b390109174f77e
FaithBench cf89797d82812c23b5d5e5c121f1d9b8983bbbce


## 3. Configure and run the comparison

- **RAGTruth:** good-quality summarization responses; any annotated span counts as positive, including `implicit_true`. This measures annotated lack of support, not necessarily factual falsity. Original train/test membership is preserved. If source IDs overlap, the overlapping training sources are removed from validation.
- **HaluEval:** each document has a benchmark-designated right and hallucinated summary. These paired labels are based on benchmark construction; right summaries are not independently verified in this notebook. Split by document to keep pairs together.
- **FaithBench:** `Questionable` or `Unwanted` (including subtypes) makes a summary positive; benign-only and unannotated summaries are negative. This mirrors the authors' example worst-severity policy, expressed here as 1 = positive. Preserve annotation categories for auditing. Split by exact source document. Results depend on this policy.

For each dataset and method, the threshold maximizes hallucination F1 on that dataset's validation sample. Test examples do not influence the threshold. These are **within-dataset** results, not a fixed-threshold cross-dataset transfer experiment. AUROC and average precision use continuous scores. Timing excludes loading, and sums single-example scoring calls; hardware and context length affect it.

The pilot samples rows after group splitting, so HaluEval test samples need not retain both members of every pair. Set limits to `None` for full pools, but full runs can take a long time. Sampled prevalence is not necessarily the population prevalence; FaithBench itself is curated for difficult cases.

In [9]:
%%writefile /content/evaluate_comparison.py
import json, random, time, hashlib, subprocess, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, confusion_matrix, f1_score
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

SEED = 42
N_VALIDATION = 50  # None = full validation pool
N_TEST = 50        # None = full test pool
OUT = Path(os.environ['FACTUALITY_RESULTS_DIR'])
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)

def read_jsonl(path):
    with open(path, encoding='utf8') as f:
        return [json.loads(line) for line in f if line.strip()]

def source_hash(text):
    return hashlib.sha256(text.encode('utf8')).hexdigest()

def row(dataset, ident, group, split, context, response, gold, extra=''):
    return dict(dataset=dataset, id=str(ident), group=str(group), split=split,
                context=context, response=response, gold=int(gold), annotation_details=extra)

def ragtruth(root):
    sources = {str(x['source_id']):x for x in read_jsonl(root/'dataset/source_info.jsonl')}
    result=[]
    for x in read_jsonl(root/'dataset/response.jsonl'):
        s=sources[str(x['source_id'])]
        if s['task_type'] != 'Summary' or x.get('quality') != 'good': continue
        context=s['source_info']; response=x['response']
        if not isinstance(context,str) or not context.strip() or not response.strip(): continue
        result.append(row('RAGTruth', x['id'], x['source_id'], x['split'], context, response,
                          bool(x['labels']), json.dumps(x['labels'], ensure_ascii=False)))
    df=pd.DataFrame(result)
    # Keep official test untouched, remove any overlapping train sources.
    overlap=set(df.loc[df.split=='train','group']) & set(df.loc[df.split=='test','group'])
    print('RAGTruth overlapping source IDs excluded from training:', len(overlap))
    train=df[(df.split=='train') & ~df.group.isin(overlap)].copy()
    test=df[df.split=='test'].copy()
    split=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    _, ix=next(split.split(train, groups=train.group))
    val=train.iloc[ix].copy(); val['split']='validation'
    return pd.concat([val,test],ignore_index=True)

def halueval(root):
    path=root/'data/summarization_data.json'
    result=[]
    for i,x in enumerate(read_jsonl(path)):
        context=x['document']
        if not context.strip(): continue
        group=source_hash(context)
        for key,label in [('right_summary',0),('hallucinated_summary',1)]:
            if x[key].strip(): result.append(row('HaluEval',f'{i}-{key}',group,'',context,x[key],label,key))
    return grouped_split(pd.DataFrame(result))

def faithbench(root):
    result=[]
    files=sorted((root/'data_for_release').glob('batch_*.json'))
    if not files: raise RuntimeError('FaithBench batch files not found')
    for path in files:
        for x in json.loads(path.read_text(encoding='utf8'))['samples']:
            context=x['source']; response=x['summary']
            if not context.strip() or not response.strip(): continue
            labels=sorted({label for a in x.get('annotations',[]) for label in a.get('label',[])})
            unknown=set(labels)-{'Benign','Questionable','Unwanted','Unwanted.Intrinsic','Unwanted.Extrinsic'}
            if unknown: raise ValueError(f'Unexpected FaithBench labels: {unknown}')
            positive=any(s == 'Questionable' or s.startswith('Unwanted') for s in labels)
            result.append(row('FaithBench',f'{path.stem}-{x["sample_id"]}',source_hash(context), '',context,response,positive,json.dumps(labels)))
    return grouped_split(pd.DataFrame(result))

def grouped_split(df):
    splitter=GroupShuffleSplit(n_splits=1,test_size=0.8,random_state=SEED)
    vi,ti=next(splitter.split(df,groups=df.group))
    val=df.iloc[vi].copy(); val['split']='validation'
    test=df.iloc[ti].copy(); test['split']='test'
    return pd.concat([val,test],ignore_index=True)

def sample(df,n):
    if n is None or n >= len(df): return df.copy().reset_index(drop=True)
    return df.sample(n=n,random_state=SEED).reset_index(drop=True)

def prepare():
    data=pd.concat([ragtruth(Path('/content/RAGTruth')),halueval(Path('/content/HaluEval')),
                    faithbench(Path('/content/FaithBench'))],ignore_index=True)
    if data.empty: raise RuntimeError('No examples loaded')
    assert set(data.gold.unique()) <= {0,1}
    chosen=[]
    for ds,part in data.groupby('dataset'):
        val=sample(part[part.split=='validation'],N_VALIDATION)
        test=sample(part[part.split=='test'],N_TEST)
        assert set(val.group).isdisjoint(set(test.group)), f'Source leakage in {ds}'
        if val.gold.nunique()!=2 or test.gold.nunique()!=2:
            raise RuntimeError(f'{ds} sample needs both labels; increase the sample size or change seed')
        print(ds, 'validation',len(val),'test',len(test), 'positive counts',val.gold.sum(),test.gold.sum())
        chosen.extend([val,test])
    return pd.concat(chosen,ignore_index=True)

def model_for(name, device):
    if name == "AlignScore-base":
        from alignscore import AlignScore
        checkpoint = hf_hub_download(repo_id="yzha/AlignScore", filename="AlignScore-base.ckpt")
        return AlignScore(model="roberta-base", batch_size=1, device=device, ckpt_path=checkpoint, evaluation_mode="nli_sp")
    elif name == "SummaC-Conv":
        from summac.model_summac import SummaCConv
        return SummaCConv(models=["vitc"], bins="percentile", granularity="sentence", nli_labels="e", device="cuda" if device.startswith("cuda") else "cpu", start_file="default", agg="mean")
    elif name == "BERTScore":
        from bert_score import BERTScorer
        return BERTScorer(model_type="roberta-large", lang="en", device=device, batch_size=1, rescale_with_baseline=False)
    elif name == "BARTScore":
        import sys
        if "/content/BARTScore" not in sys.path:
            sys.path.insert(0, "/content/BARTScore")
        from bart_score import BARTScorer
        return BARTScorer(device=device, checkpoint="facebook/bart-large-cnn")
    else:
        raise ValueError(f"Unknown method: {name}")

def score_all(df,name,device):
    model=model_for(name,device)
    path=OUT / (name.replace('-','_')+'_scores.csv')
    output=[]
    for item in tqdm(df.to_dict('records'),desc=name):
        if device.startswith('cuda'): torch.cuda.synchronize()
        start=time.perf_counter()
        with torch.inference_mode():
            if name == "AlignScore-base":
                score = float(model.score(contexts=[item["context"]], claims=[item["response"]])[0])
                # Higher AlignScore means more consistent -> lower hallucination
                hallucination_score = 1.0 - score
            elif name == "SummaC-Conv":
                score = float(model.score([item["context"]], [item["response"]])["scores"][0])
                # Higher SummaC means more consistent -> lower hallucination
                hallucination_score = 1.0 - score
            elif name == "BERTScore":
                _, _, f1 = model.score(cands=[item["response"]], refs=[item["context"]], batch_size=1)
                score = float(f1[0].item())
                hallucination_score = 1.0 - score
            elif name == "BARTScore":
                score = float(model.score([item["context"]], [item["response"]], batch_size=1)[0])
                # BARTScore outputs log probability (negative, e.g. -2.5). Higher (less negative) is more consistent.
                # We negate it so that higher values indicate higher hallucination.
                hallucination_score = -score
            else:
                raise ValueError(f"Unknown method: {name}")
        if device.startswith('cuda'): torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        output.append({
            **item,
            "method": name,
            "consistency_score": score,
            "hallucination_score": hallucination_score,
            "seconds": seconds
        })
    result=pd.DataFrame(output); result.to_csv(path,index=False)
    del model
    if device.startswith('cuda'): torch.cuda.empty_cache()
    return result

def assess(all_scores):
    summaries=[]; predictions=[]
    for (method,ds),g in all_scores.groupby(['method','dataset']):
        val=g[g.split=='validation']; test=g[g.split=='test'].copy()
        values=np.sort(val.hallucination_score.unique())
        candidates=np.r_[np.nextafter(values[0],-np.inf),values,np.nextafter(values[-1],np.inf)]
        threshold=float(max(candidates,key=lambda t:f1_score(val.gold,val.hallucination_score>=t,zero_division=0)))
        test['predicted']=(test.hallucination_score>=threshold).astype(int)
        y=test.gold.to_numpy(); pred=test.predicted.to_numpy(); s=test.hallucination_score.to_numpy()
        precision,recall,f1,_=precision_recall_fscore_support(y,pred,average='binary',zero_division=0)
        cm=confusion_matrix(y,pred,labels=[0,1])
        tn,fp,fn,tp=cm.ravel()
        assert tp+fn==int(y.sum()) and tn+fp==int((1-y).sum())
        summaries.append(dict(dataset=ds,method=method,n=len(test),positive_count=int(y.sum()),
            prevalence=float(y.mean()),threshold=threshold,auroc=float(roc_auc_score(y,s)),
            average_precision=float(average_precision_score(y,s)),precision=float(precision),recall=float(recall),
            f1=float(f1),mean_seconds=float(test.seconds.mean()),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)))
        predictions.append(test)
    return pd.DataFrame(summaries),pd.concat(predictions,ignore_index=True)

def main():
    df=prepare(); df.to_csv(OUT/'selected_examples.csv',index=False)
    device='cuda:0' if torch.cuda.is_available() else 'cpu'; print('Device:',device)
    methods = ["AlignScore-base","SummaC-Conv","BERTScore","BARTScore"]
    scores = pd.concat(
    [score_all(df, method, device) for method in methods],
    ignore_index=True,)
    summary,preds=assess(scores)
    summary.to_csv(OUT/'metrics.csv',index=False); preds.to_csv(OUT/'test_predictions.csv',index=False)
    preds[preds.gold!=preds.predicted].to_csv(OUT/'test_errors.csv',index=False)
    metadata={'seed':SEED,'validation_limit':N_VALIDATION,'test_limit':N_TEST,'device':device,
      'threshold':'per dataset and method, maximizing validation F1',
      'score_direction':'negative consistency score; higher means more likely hallucination',
      'label_policies':{'RAGTruth':'any annotated span, including implicit_true',
       'HaluEval':'right_summary=0; hallucinated_summary=1 (dataset-designated)',
       'FaithBench':'worst severity: Questionable or Unwanted=1; Benign-only or no annotations=0'},
      'dataset_commits':{name:subprocess.check_output(['git','-C',f'/content/{name}','rev-parse','HEAD'],text=True).strip() for name in ['RAGTruth','HaluEval','FaithBench']}}
    (OUT/'settings.json').write_text(json.dumps(metadata,indent=2))
    print('\nTEST RESULTS\n',summary.to_string(index=False)); print('\nSaved to',OUT)

if __name__=='__main__': main()

Overwriting /content/evaluate_comparison.py


In [7]:
# Execute in the isolated environment and stream progress.
import os
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

process = subprocess.Popen([PYTHON, '-u', '/content/evaluate_comparison.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in process.stdout:
    print(line, end='')
if process.wait() != 0:
    raise RuntimeError('Evaluation stopped. Read the error above; partial scores may be saved.')

RAGTruth overlapping source IDs excluded from training: 0
FaithBench validation 50 test 50 positive counts 31 37
HaluEval validation 50 test 50 positive counts 26 24
RAGTruth validation 50 test 50 positive counts 16 9
Device: cuda:0
/content/alignscore-env/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expecte

RuntimeError: Evaluation stopped. Read the error above; partial scores may be saved.

## 4. View and download results
The Drive folder and ZIP contain selected examples, validation and test scores, test predictions, errors, metrics, environment, and dataset versions. Scores from the two models are evaluated on identical selected examples. Review false positives and false negatives before drawing conclusions.

In [ ]:
import pandas as pd, shutil
from google.colab import files
RESULTS = RUN_DIR
display(pd.read_csv(RESULTS / 'metrics.csv'))
display(pd.read_csv(RESULTS / 'test_errors.csv').head())
with open(RESULTS / 'environment.txt', 'w') as f:
    subprocess.run(['uv','pip','freeze','--python',PYTHON], stdout=f, check=True)
shutil.copy('/content/evaluate_comparison.py', RESULTS / 'evaluate_comparison.py')
zip_path = shutil.make_archive(str(RUN_DIR.parent / RUN_DIR.name), 'zip', root_dir=RESULTS)
print('Results folder:', RUN_DIR)
print('ZIP saved to Drive:', zip_path)
files.download(zip_path)  # Optional: also download to your computer

,dataset,method,n,positive_count,prevalence,threshold,auroc,average_precision,precision,recall,f1,mean_seconds,tn,fp,fn,tp
0,FaithBench,AlignScore-base,50,37,0.74,-0.991751,0.625780,0.833235,0.750000,0.972973,0.847059,0.365750,1,12,1,36
1,HaluEval,AlignScore-base,50,24,0.48,-0.762587,0.692308,0.744311,0.529412,0.750000,0.620690,0.460977,10,16,6,18
2,RAGTruth,AlignScore-base,50,9,0.18,-0.683371,0.712737,0.351949,0.333333,0.666667,0.444444,0.695049,29,12,3,6
3,FaithBench,SummaC-Conv,50,37,0.74,-0.526094,0.641372,0.802028,0.800000,0.756757,0.777778,1.096393,6,7,9,28
4,HaluEval,SummaC-Conv,50,24,0.48,-0.406066,0.786859,0.721551,0.600000,0.875000,0.711864,1.446428,12,14,3,21
5,RAGTruth,SummaC-Conv,50,9,0.18,-0.381940,0.466125,0.172581,0.205882,0.777778,0.325581,2.290839,14,27,2,7


,dataset,id,group,split,context,response,gold,annotation_details,method,consistency_score,hallucination_score,seconds,predicted
0,FaithBench,batch_4-39,a7c07d3d6d75803986dee937533bb235d8a20e35ed96da...,test,"The Dons loaned Smith, 21, to United in July a...",The Dons loaned 21-year-old Smith to United in...,1,"[""Questionable""]",AlignScore-base,0.998432,-0.998432,0.160143,0
1,FaithBench,batch_14-41,fe3cc5b4a906def96dddfeb8221a5cdebc4dee51de0a51...,test,Little is beyond their reach. Official documen...,The passage discusses the activities of three...,0,[],AlignScore-base,0.758918,-0.758918,1.034533,1
2,FaithBench,batch_2-4,9f595904c50cdee5124f4faa1d330f0feaf9a3e5250c9b...,test,Veeram ( Valour ) is a 2014 Indian Tamil actio...,The passage describes two different films titl...,0,[],AlignScore-base,0.860019,-0.860019,0.225111,1
3,FaithBench,batch_3-19,47ddb2a902d4235a8ab906833d38fe62c389a9d10a4acf...,test,Lake Providence is a town in and the parish se...,Lake Providence is a town and the parish seat ...,0,[],AlignScore-base,0.986278,-0.986278,0.080772,1
4,FaithBench,batch_11-49,89c9cdcd6692fdaf2e407bde12d04df51095303998a0ee...,test,a species of whale that was feared to be extin...,"A species of whale, feared extinct, was found ...",0,[],AlignScore-base,0.652877,-0.652877,0.397476,1


Results folder: /content/drive/MyDrive/EECS_6412_factuality_results/run_20261027_202312_806888
ZIP saved to Drive: /content/drive/MyDrive/EECS_6412_factuality_results/run_20261027_202312_806888.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import pandas as pd, shutil
from google.colab import files
RESULTS = RUN_DIR
display(pd.read_csv(RESULTS / 'metrics.csv'))
display(pd.read_csv(RESULTS / 'test_errors.csv').head())
with open(RESULTS / 'environment.txt', 'w') as f:
    subprocess.run(['uv','pip','freeze','--python',PYTHON], stdout=f, check=True)
shutil.copy('/content/evaluate_comparison.py', RESULTS / 'evaluate_comparison.py')
zip_path = shutil.make_archive(str(RUN_DIR.parent / RUN_DIR.name), 'zip', root_dir=RESULTS)
print('Results folder:', RUN_DIR)
print('ZIP saved to Drive:', zip_path)
files.download(zip_path)  # Optional: also download to your computer

,dataset,method,n,positive_count,prevalence,threshold,auroc,average_precision,precision,recall,f1,mean_seconds,tn,fp,fn,tp
0,FaithBench,AlignScore-base,50,37,0.74,0.008249,0.625780,0.833235,0.750000,0.972973,0.847059,0.390430,1,12,1,36
1,HaluEval,AlignScore-base,50,24,0.48,0.237413,0.692308,0.744311,0.529412,0.750000,0.620690,0.530710,10,16,6,18
2,RAGTruth,AlignScore-base,50,9,0.18,0.316629,0.712737,0.351949,0.333333,0.666667,0.444444,0.741186,29,12,3,6
3,FaithBench,BARTScore,50,37,0.74,1.096386,0.702703,0.844439,0.795455,0.945946,0.864198,0.097745,4,9,2,35
4,HaluEval,BARTScore,50,24,0.48,1.750797,0.724359,0.735781,0.633333,0.791667,0.703704,0.156587,15,11,5,19
5,RAGTruth,BARTScore,50,9,0.18,2.209720,0.474255,0.230688,0.250000,0.222222,0.235294,0.153630,35,6,7,2
6,FaithBench,BERTScore,50,37,0.74,0.055780,0.540541,0.751816,0.760870,0.945946,0.843373,0.089031,2,11,2,35
7,HaluEval,BERTScore,50,24,0.48,0.140059,0.495192,0.490420,0.472222,0.708333,0.566667,0.124736,7,19,7,17
8,RAGTruth,BERTScore,50,9,0.18,0.143419,0.577236,0.322881,0.187500,0.333333,0.240000,0.140171,28,13,6,3
9,FaithBench,SummaC-Conv,50,37,0.74,0.473906,0.641372,0.802028,0.800000,0.756757,0.777778,1.176764,6,7,9,28


,dataset,id,group,split,context,response,gold,annotation_details,method,consistency_score,hallucination_score,seconds,predicted
0,FaithBench,batch_4-39,a7c07d3d6d75803986dee937533bb235d8a20e35ed96da...,test,"The Dons loaned Smith, 21, to United in July a...",The Dons loaned 21-year-old Smith to United in...,1,"[""Questionable""]",AlignScore-base,0.998432,0.001568,0.168149,0
1,FaithBench,batch_14-41,fe3cc5b4a906def96dddfeb8221a5cdebc4dee51de0a51...,test,Little is beyond their reach. Official documen...,The passage discusses the activities of three...,0,[],AlignScore-base,0.758918,0.241082,1.070588,1
2,FaithBench,batch_2-4,9f595904c50cdee5124f4faa1d330f0feaf9a3e5250c9b...,test,Veeram ( Valour ) is a 2014 Indian Tamil actio...,The passage describes two different films titl...,0,[],AlignScore-base,0.860019,0.139981,0.124106,1
3,FaithBench,batch_3-19,47ddb2a902d4235a8ab906833d38fe62c389a9d10a4acf...,test,Lake Providence is a town in and the parish se...,Lake Providence is a town and the parish seat ...,0,[],AlignScore-base,0.986278,0.013722,0.084600,1
4,FaithBench,batch_11-49,89c9cdcd6692fdaf2e407bde12d04df51095303998a0ee...,test,a species of whale that was feared to be extin...,"A species of whale, feared extinct, was found ...",0,[],AlignScore-base,0.652877,0.347123,0.278823,1


Results folder: /content/drive/MyDrive/EECS_6412_factuality_results/run_20261028_171130_261670
ZIP saved to Drive: /content/drive/MyDrive/EECS_6412_factuality_results/run_20261028_171130_261670.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Original notebook (reference only)
The following cells are preserved from your upload. They run the earlier RAGTruth-only experiment. Do not run them during the new comparison: they use a separate workflow and can overwrite or alter the environment.

In [ ]:
# Run in a fresh Colab runtime.
!pip -q install uv

import subprocess
import sys
from pathlib import Path

ENV = Path("/content/alignscore-env")
PYTHON = str(ENV / "bin/python")

def run(command):
    subprocess.run(command, check=True)

# Create the venv securely using Python's built-in venv module instead of uv venv
run([sys.executable, "-m", "venv", str(ENV)])

run([
    "uv", "pip", "install", "--python", PYTHON,
    "torch==1.13.1",
    "numpy==1.24.4",
    "transformers==4.30.2",
    "pytorch-lightning==1.9.5",
    "torchmetrics==0.11.4",
    "spacy==3.7.5",
    "nltk==3.8.1",
    "scikit-learn==1.3.2",
    "pandas==2.0.3",
    "huggingface-hub==0.25.2",
    "protobuf==3.20.0",
    "tensorboard==2.14.0",
    "datasets==2.14.7",
    "pyarrow==14.0.2",
    "jsonlines==2.0.0",
    "tqdm",
])

run([
    "uv", "pip", "install", "--python", PYTHON,
    "--no-deps",
    "git+https://github.com/yuh-zha/AlignScore.git",
])

# Install the spaCy model directly using uv to avoid internal pip dependency issues
run([
    "uv", "pip", "install", "--python", PYTHON,
    "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.0/en_core_web_sm-3.7.0-py3-none-any.whl"
])

run([
    PYTHON, "-c",
    "import nltk; nltk.download('punkt', quiet=True)"
])

print("Installation complete.")

In [ ]:
import subprocess

RAG_DIR = Path("/content/RAGTruth")

if not RAG_DIR.exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/ParticleMedia/RAGTruth.git",
        str(RAG_DIR),
    ])

DATA_COMMIT = subprocess.check_output(
    ["git", "-C", str(RAG_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("RAGTruth commit:", DATA_COMMIT)
print("Dataset files:")
for path in RAG_DIR.rglob("*.jsonl"):
    print(path)

In [ ]:
%%writefile /content/evaluate_alignscore.py
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from alignscore import AlignScore

# ---------- Configuration ----------
SEED = 42
N_VALIDATION = 50
N_TEST = 50             # Set to None to evaluate all eligible test responses.
BATCH_SIZE = 1
TASKS = {"QA", "Summary"}  # Start with text evidence; exclude Data2txt initially.
OUT = Path("/content/alignscore_results")
OUT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

# ---------- Load dataset ----------
def find_unique(filename):
    matches = list(Path("/content/RAGTruth").rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {filename}, found: {matches}")
    return matches[0]

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

sources = {
    str(x["source_id"]): x
    for x in read_jsonl(find_unique("source_info.jsonl"))
}
responses = read_jsonl(find_unique("response.jsonl"))

def evidence(source):
    info = source["source_info"]
    if source["task_type"] == "Summary":
        assert isinstance(info, str)
        return info
    if source["task_type"] == "QA":
        # The question is not treated as evidence.
        passages = info["passages"]
        assert isinstance(passages, str)
        return passages
    raise ValueError(source["task_type"])

rows = []
for response in responses:
    source_id = str(response["source_id"])
    source = sources[source_id]

    if source["task_type"] not in TASKS:
        continue
    # Explicitly limit this pilot to good-quality responses.
    if response["quality"] != "good":
        continue

    context = evidence(source)
    claim = response["response"]
    if not context.strip() or not claim.strip():
        continue

    rows.append({
        "id": str(response["id"]),
        "source_id": source_id,
        "split": response["split"],
        "task": source["task_type"],
        "generator": response["model"],
        "context": context,
        "response": claim,
        "gold": int(len(response["labels"]) > 0),
    })

df = pd.DataFrame(rows)
assert not df.empty

train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()
assert not train.empty and not test.empty

# Keep source documents separate across development and test.
assert set(train.source_id).isdisjoint(set(test.source_id))

# Create a validation pool from training sources, not test responses.
train_source_ids = sorted(train.source_id.unique())
_, val_source_ids = train_test_split(
    train_source_ids, test_size=0.20, random_state=SEED
)
validation = train[train.source_id.isin(val_source_ids)].copy()

def sample_rows(frame, n):
    if n is None or n >= len(frame):
        return frame.reset_index(drop=True)
    # Stratify by label when possible; retain approximate natural prevalence.
    if frame.gold.nunique() == 2 and frame.gold.value_counts().min() >= 2:
        selected, _ = train_test_split(
            frame,
            train_size=n,
            stratify=frame.gold,
            random_state=SEED,
        )
        return selected.reset_index(drop=True)
    return frame.sample(n=n, random_state=SEED).reset_index(drop=True)

validation = sample_rows(validation, N_VALIDATION)
test = sample_rows(test, N_TEST)

for name, frame in [("validation", validation), ("test", test)]:
    print(f"\n{name}: {len(frame)} responses")
    print(pd.crosstab(frame.task, frame.gold))
    frame.drop(columns=["context", "response"]).to_csv(
        OUT / f"{name}_selection.csv", index=False
    )

# ---------- Load pretrained AlignScore ----------
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("\nDevice:", device)

checkpoint = hf_hub_download(
    repo_id="yzha/AlignScore",
    filename="AlignScore-base.ckpt",
)

scorer = AlignScore(
    model="roberta-base",
    batch_size=BATCH_SIZE,
    device=device,
    ckpt_path=checkpoint,
    evaluation_mode="nli_sp",
)

# Smoke test: inspect these scores; do not use them for threshold selection.
print("\nSmoke-test consistency scores:")
print(scorer.score(
    contexts=[
        "The study enrolled 34 participants.",
        "The study enrolled 34 participants.",
    ],
    claims=[
        "The study enrolled 34 participants.",
        "The study enrolled 500 participants.",
    ],
))

# ---------- Score and save incrementally ----------
def score_frame(frame, name):
    path = OUT / f"{name}_predictions.csv"
    scored = []

    for row in tqdm(
        frame.to_dict("records"), desc=f"Scoring {name}"
    ):
        if device.startswith("cuda"):
            torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            consistency = float(scorer.score(
                contexts=[row["context"]],
                claims=[row["response"]],
            )[0])

        if device.startswith("cuda"):
            torch.cuda.synchronize()
        seconds = time.perf_counter() - start

        if not np.isfinite(consistency):
            raise RuntimeError(f"Non-finite score for {row['id']}")

        scored.append({
            **row,
            "consistency_score": consistency,
            "hallucination_score": 1.0 - consistency,
            "seconds": seconds,
        })
        # Save after every response.
        pd.DataFrame(scored).to_csv(path, index=False)

    return pd.DataFrame(scored)

val_scores = score_frame(validation, "validation")
test_scores = score_frame(test, "test")

# ---------- Select threshold using validation ONLY ----------
if val_scores.gold.nunique() < 2:
    raise RuntimeError(
        "Validation sample needs both labels. Increase N_VALIDATION."
    )

thresholds = np.unique(np.concatenate([
    [-1e-6],
    val_scores.hallucination_score.to_numpy(),
    [1.000001],
]))

def f1_at(threshold):
    predictions = (
        val_scores.hallucination_score.to_numpy() >= threshold
    ).astype(int)
    return precision_recall_fscore_support(
        val_scores.gold, predictions,
        average="binary", zero_division=0,
    )[2]

threshold = float(max(thresholds, key=f1_at))
print("\nValidation-selected threshold:", threshold)

# ---------- Evaluate held-out test sample ----------
def metrics(frame):
    y = frame.gold.to_numpy()
    scores = frame.hallucination_score.to_numpy()
    predicted = (scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, predicted, average="binary", zero_division=0
    )

    both_labels = len(np.unique(y)) == 2
    return {
        "n": len(frame),
        "hallucination_prevalence": float(y.mean()),
        "threshold": threshold,
        "auroc": float(roc_auc_score(y, scores)) if both_labels else None,
        "average_precision": (
            float(average_precision_score(y, scores))
            if both_labels else None
        ),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "mean_seconds_per_response": float(frame.seconds.mean()),
        "confusion_matrix_labels_0_1": confusion_matrix(
            y, predicted, labels=[0, 1]
        ).tolist(),
    }

results = {
    "overall": metrics(test_scores),
    "by_task": {
        task: metrics(group)
        for task, group in test_scores.groupby("task")
    },
    "settings": {
        "seed": SEED,
        "model": "AlignScore-base",
        "mode": "nli_sp",
        "device": device,
        "tasks": sorted(TASKS),
        "quality_filter": "good",
        "label_policy": "any annotated span, including implicit_true",
    },
}

test_scores["predicted"] = (
    test_scores.hallucination_score >= threshold
).astype(int)
test_scores.to_csv(OUT / "test_predictions.csv", index=False)
test_scores[test_scores.gold != test_scores.predicted].to_csv(
    OUT / "test_errors.csv", index=False
)

with open(OUT / "metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nTEST RESULTS")
print(json.dumps(results, indent=2))
print("\nFiles saved to:", OUT)

In [ ]:
import subprocess
from pathlib import Path

# Define explicit paths
ENV = Path("/content/alignscore-env")
PYTHON = str(ENV / "bin/python")

# Install an older version of setuptools that contains pkg_resources
subprocess.run([
    "uv", "pip", "install", "--python", PYTHON, "--force-reinstall", "setuptools==65.5.0"
], check=True)

# Stream output while the script runs.
process = subprocess.Popen(
    [PYTHON, "-u", "/content/evaluate_alignscore.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="")

if process.wait() != 0:
    raise RuntimeError("Evaluation failed. Check the error printed above.")

In [ ]:
import json
import pandas as pd
import shutil
from google.colab import files

RESULTS = Path("/content/alignscore_results")

with open(RESULTS / "metrics.json") as f:
    metrics = json.load(f)

display(pd.DataFrame([metrics["overall"]]))
display(pd.read_csv(RESULTS / "test_errors.csv").head())

# Record the dataset version and installed packages.
(RESULTS / "ragtruth_commit.txt").write_text(DATA_COMMIT)
with open(RESULTS / "environment.txt", "w") as f:
    subprocess.run(
        ["uv", "pip", "freeze", "--python", PYTHON],
        stdout=f, check=True,
    )

shutil.make_archive(
    "/content/alignscore_results", "zip", root_dir=RESULTS
)
files.download("/content/alignscore_results.zip")